# KMOS Pipeline — Stacked Galaxy Showcase

Groups all IFUs across every OB in `kmos_ob_proc_server/` by **group_id**
(from `scripts/table_ifus_ECDFS 2.fits` and `scripts/table_ifus_COSMOS_Field_exptime.fits`).
IFUs sharing the same group_id point at the same galaxy; their individual
300 s exposures are co-added by a per-pixel **nanmedian** across all OBs.

**Sections**
1. Imports and configuration
2. Helper functions
3. Load group_id catalog from FITS tables
4. Load all IFUs — stack by group_id
5. Ranking table
6. Line-window gallery — Hα · Hβ · [OIII]5007 · [SII]6716
7. Detail view — full spectrum + spatial maps

In [11]:
import glob, os, csv, pathlib, warnings
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from astropy.io import fits

# ── paths ─────────────────────────────────────────────────────────────────────
BASE         = "kmos_ob_proc_server"
SUFFIX       = "_s_i_bkg2d_rv_tell.fits"
ECDFS_TABLE  = pathlib.Path("scripts") / "table_ifus_ECDFS 2.fits"
COSMOS_TABLE = pathlib.Path("scripts") / "table_ifus_COSMOS_Field_exptime.fits"

# ── extraction parameters ─────────────────────────────────────────────────────
APERTURE          = 5
EDGE_MARGIN       = 4
MAX_CENTER_OFFSET = 4.0

# ── Hα ───────────────────────────────────────────────────────────────────────
HA_REST      = 0.65628   # µm
SNR_WIN_HALF = 0.0025
SNR_CONT_IN  = 0.006
SNR_CONT_OUT = 0.022
SNR_MIN      = 2.0

# ── display ───────────────────────────────────────────────────────────────────
TOP_DETAIL   = 8          # galaxies in detail view
ZOOM_HALF    = 0.018      # ±µm around each line in zoom panels
HB_ZOOM_HALF = 0.012      # slightly narrower window for Hβ (often weaker)

PALETTE = [
    "#1a6faf", "#c03a2b", "#27ae60", "#8e44ad", "#d35400",
    "#16a085", "#2c3e50", "#e74c3c", "#2980b9", "#1abc9c",
    "#f39c12", "#7f8c8d", "#6c3483", "#1e8449", "#922b21",
    "#0e6655", "#784212", "#1b4f72", "#f1c40f", "#8e44ad",
    "#3498db", "#e67e22", "#2ecc71", "#9b59b6", "#e74c3c",
    "#34495e", "#f39c12", "#2c3e50", "#d35400", "#16a085",
    "#2980b9", "#c0392b", "#27ae60", "#8e44ad", "#f1c40f",
    "#7f8c8d", "#1abc9c", "#e67e22", "#2c3e50", "#3498db",
    "#8e44ad", "#d35400", "#16a085", "#c03a2b", "#27ae60",
    "#922b21", "#1e8449", "#1b4f72", "#784212", "#0e6655",
]

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 9,
    "axes.titlesize": 9, "axes.titleweight": "bold", "axes.titlepad": 5,
    "axes.labelsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#444", "axes.linewidth": 0.8,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": "#bbb", "grid.alpha": 0.3, "grid.linewidth": 0.5,
    "figure.facecolor": "white", "savefig.facecolor": "white",
    "figure.dpi": 120,
})

In [ ]:
# ── data helpers ──────────────────────────────────────────────────────────────
def wavelength_axis(hdr):
    return (np.arange(hdr["NAXIS3"]) + 1 - hdr["CRPIX3"]) * hdr["CDELT3"] + hdr["CRVAL3"]

def active_ifus(hdul):
    result = []
    for i in range(1, 25):
        try:
            d = hdul[i].data
            if d is not None and np.ndim(d) == 3:
                result.append(i)
        except Exception:
            pass
    return result

def _erode_valid(finite, margin):
    inv = ~finite
    eroded = inv.copy()
    for s in range(1, margin + 1):
        eroded[s:]  |= inv[:-s]
        eroded[:-s] |= inv[s:]
    return ~eroded

def aperture_spectrum(hdul, ifu, center, box=APERTURE, margin=EDGE_MARGIN):
    cube  = np.array(hdul[ifu].data)
    hdr   = hdul[ifu].header
    py, px = center
    h2    = box // 2
    y0, y1 = max(py - h2, 0), min(py + h2 + 1, cube.shape[1])
    x0, x1 = max(px - h2, 0), min(px + h2 + 1, cube.shape[2])
    patch  = cube[:, y0:y1, x0:x1].reshape(cube.shape[0], -1)
    valid  = _erode_valid(np.isfinite(patch), margin)
    cleaned = np.where(valid, patch, np.nan)
    rows_ok = valid.any(axis=1)
    spec    = np.full(cube.shape[0], np.nan)
    spec[rows_ok] = np.nansum(cleaned[rows_ok], axis=1)
    return wavelength_axis(hdr), spec

def ifu_center(cube):
    return (cube.shape[1] // 2, cube.shape[2] // 2)

def find_exposures(base=BASE, suffix=SUFFIX):
    result = defaultdict(list)
    for path in sorted(glob.glob(os.path.join(base, "*", "science", f"*{suffix}"))):
        parts = path.replace("\\", "/").split("/")
        ob    = parts[-3]
        with fits.open(path) as h:
            band = h[0].header.get("HIERARCH ESO INS GRAT1 ID", "??")
        result[(ob, band)].append(path)
    return dict(result)

def ob_to_dpid(ob_name):
    """Convert OB folder name to dp_id format.
    'KMOS.2026-01-13T03-42-28.829' → 'KMOS.2026-01-13T03:42:28.829'
    """
    if 'T' not in ob_name:
        return ob_name
    date_part, time_part = ob_name.split('T', 1)
    parts = time_part.split('-', 2)
    if len(parts) == 3:
        return f"{date_part}T{parts[0]}:{parts[1]}:{parts[2]}"
    return ob_name

def ha_peak_center(cube, wave, z, margin=EDGE_MARGIN, min_nbr=0.08):
    ha_obs = HA_REST * (1.0 + z)
    hp     = int(np.argmin(np.abs(wave - ha_obs)))
    p0, p1 = max(hp - 3, 0), min(hp + 4, cube.shape[0])
    valid  = _erode_valid(np.isfinite(cube[p0:p1]), margin)
    ha_im  = np.nansum(np.where(valid, cube[p0:p1], np.nan), axis=0)
    ny, nx = ha_im.shape
    flat   = ha_im.ravel()
    order  = np.argsort(np.where(np.isfinite(flat), flat, -np.inf))[::-1]
    for idx in order:
        v = flat[idx]
        if not np.isfinite(v) or v <= 0:
            break
        py, px = np.unravel_index(idx, (ny, nx))
        nbrs = [ha_im[py+dy, px+dx]
                for dy in (-1, 0, 1) for dx in (-1, 0, 1)
                if (dy, dx) != (0, 0) and 0 <= py+dy < ny and 0 <= px+dx < nx]
        nbrs = [n for n in nbrs if np.isfinite(n)]
        if nbrs and np.mean(nbrs) >= min_nbr * v:
            return (py, px), ha_im
    return ifu_center(cube), ha_im

def line_snr(wave, spec, z, line_rest,
             win_half=SNR_WIN_HALF, cont_in=SNR_CONT_IN, cont_out=SNR_CONT_OUT,
             min_width_px=3):
    """Continuum-subtracted peak SNR; requires ≥ min_width_px pixels above noise to reject spikes."""
    lam = line_rest * (1.0 + z)
    if not (wave.min() < lam < wave.max()):
        return np.nan
    sig_mask  = np.abs(wave - lam) <= win_half
    cont_mask = (np.abs(wave - lam) >= cont_in) & (np.abs(wave - lam) <= cont_out)
    if sig_mask.sum() < 1 or cont_mask.sum() < 8:
        return np.nan
    wc, sc = wave[cont_mask], spec[cont_mask]
    ok = np.isfinite(sc)
    if ok.sum() < 4:
        return np.nan
    p         = np.polyfit(wc[ok], sc[ok], 1)
    cont_line = np.polyval(p, wave)
    peak      = np.nanmax(spec[sig_mask] - cont_line[sig_mask])
    cs_cont   = (spec[cont_mask] - cont_line[cont_mask])[np.isfinite(spec[cont_mask])]
    if cs_cont.size < 4:
        return np.nan
    mad   = np.nanmedian(np.abs(cs_cont - np.nanmedian(cs_cont)))
    noise = 1.4826 * mad
    if noise <= 0 or not np.isfinite(noise):
        return np.nan
    # Require at least min_width_px spectral pixels above 1-sigma to reject single-pixel spikes
    cs_sig = spec[sig_mask] - cont_line[sig_mask]
    if np.sum(np.isfinite(cs_sig) & (cs_sig > noise)) < min_width_px:
        return np.nan
    return float(peak / noise)

def ha_snr(wave, spec, z):
    return line_snr(wave, spec, z, HA_REST)

def narrowband_image(cube, wave, z, line_rest, margin=EDGE_MARGIN):
    lam_obs = line_rest * (1.0 + z)
    if not (wave.min() < lam_obs < wave.max()):
        return None
    hp      = int(np.argmin(np.abs(wave - lam_obs)))
    p0, p1  = max(hp - 3, 0), min(hp + 4, cube.shape[0])
    valid   = _erode_valid(np.isfinite(cube[p0:p1]), margin)
    return np.nansum(np.where(valid, cube[p0:p1], np.nan), axis=0)

# ── emission lines ────────────────────────────────────────────────────────────
EMISSION_LINES = {
    "[OII]3727":    (0.37270, "#e377c2"),
    "Hβ":           (0.48627, "#1f77b4"),
    "[OIII]4959":   (0.49594, "#7fdbff"),
    "[OIII]5007":   (0.50082, "#17becf"),
    "Hα":           (0.65628, "#d62728"),
    "[NII]6583":    (0.65835, "#ff7f0e"),
    "[SII]6716":    (0.67163, "#9467bd"),
    "[SII]6731":    (0.67308, "#c5b0d8"),
}

# ── redshift catalogue ────────────────────────────────────────────────────────
_Z_TABLE, _Z_TYPE, _Z_QUALITY = {}, {}, {}
_Z_CSV = pathlib.Path("scripts") / "target_redshifts.csv"
if _Z_CSV.exists():
    with open(_Z_CSV, newline="", encoding="utf-8") as _f:
        for _row in csv.DictReader(_f):
            if (_row.get("redshift") or "").strip():
                _key = (_row["field"].strip(), _row["arm_target_id"].strip())
                _Z_TABLE[_key] = float(_row["redshift"].strip())
                _Z_TYPE[_key]  = (_row.get("redshift_type") or "").strip()
                _q = (_row.get("quality") or "").strip()
                try:
                    _Z_QUALITY[_key] = float(_q)
                except (ValueError, TypeError):
                    _Z_QUALITY[_key] = np.nan
    print(f"Loaded {len(_Z_TABLE)} redshifts  "
          f"(spec: {sum(1 for v in _Z_TYPE.values() if v=='spec')}, "
          f"photo: {sum(1 for v in _Z_TYPE.values() if v=='photo')}, "
          f"cluster: {sum(1 for v in _Z_TYPE.values() if v=='cluster')})")
else:
    print(f"WARNING: {_Z_CSV} not found — emission lines will not be drawn")

def get_z_meta(hdul, ifu):
    hdr0  = hdul[0].header
    field = (hdr0.get("ESO OBS TARG NAME") or "").strip()
    key   = f"ESO OCS ARM{ifu} NAME"
    tid   = str(hdr0.get(key) or hdr0.get(f"HIERARCH {key}") or "").strip()
    if not field or not tid:
        return None, "", np.nan
    k = (field, tid)
    return _Z_TABLE.get(k), _Z_TYPE.get(k, ""), _Z_QUALITY.get(k, np.nan)

In [ ]:
# ── Load FITS catalog tables → group_id lookup ─────────────────────────────────
# dpid_ifu_to_group[(dp_id, ifu)] = group_id
dpid_ifu_to_group = {}

for tbl_path in [ECDFS_TABLE, COSMOS_TABLE]:
    if not tbl_path.exists():
        print(f"WARNING: {tbl_path} not found")
        continue
    with fits.open(tbl_path) as hdul:
        t = hdul[1].data
        for i in range(len(t)):
            dpid_ifu_to_group[(t['dp_id'][i].strip(), int(t['ifu'][i]))] = int(t['group_id'][i])
    print(f"Loaded {len(t):,} rows from {tbl_path.name}")

print(f"Total catalog entries from tables: {len(dpid_ifu_to_group):,}")

# ── Fallback: build synthetic group_ids for fields not in the FITS tables ──────
# Fields like SPT2040 and XMM2215 have no ECDFS/COSMOS catalog table; assign
# synthetic group_ids (≥ 90000) from (field, arm_target_id) pairs found in Z_TABLE.
_synthetic_next = 90000
_field_tid_to_group = {}   # (field, arm_target_id) → synthetic group_id

for path in sorted(glob.glob(os.path.join(BASE, "*", "science", f"*{SUFFIX}"))):
    parts = path.replace("\\", "/").split("/")
    ob    = parts[-3]
    dpid  = ob_to_dpid(ob)
    with fits.open(path) as h:
        hdr0  = h[0].header
        field = (hdr0.get("ESO OBS TARG NAME") or "").strip()
        if not field:
            continue
        for ifu in range(1, 25):
            if (dpid, ifu) in dpid_ifu_to_group:
                continue           # already covered by tables
            if ifu >= len(h):
                continue
            try:
                d = h[ifu].data
                if d is None or np.ndim(d) != 3:
                    continue
            except Exception:
                continue
            key_hdr = f"ESO OCS ARM{ifu} NAME"
            tid = str(hdr0.get(key_hdr) or hdr0.get(f"HIERARCH {key_hdr}") or "").strip()
            if not tid:
                continue
            k = (field, tid)
            if k not in _Z_TABLE:
                continue           # unknown target, skip
            if k not in _field_tid_to_group:
                _field_tid_to_group[k] = _synthetic_next
                _synthetic_next += 1
            dpid_ifu_to_group[(dpid, ifu)] = _field_tid_to_group[k]

if _field_tid_to_group:
    print(f"Added {len(_field_tid_to_group)} synthetic group_ids for targets not in catalog tables "
          f"(ids {90000}–{_synthetic_next - 1})")
    from collections import Counter
    field_counts = Counter(k[0] for k in _field_tid_to_group)
    for field, n in field_counts.most_common():
        print(f"  {field}: {n} targets")

print(f"Total catalog entries (incl. synthetic): {len(dpid_ifu_to_group):,}")

# Reverse map for diagnostics
group_to_dpids = defaultdict(list)
for (dpid, ifu), gid in dpid_ifu_to_group.items():
    group_to_dpids[gid].append((dpid, ifu))
print(f"Unique group_ids: {len(group_to_dpids):,}")

In [14]:
# ── Load all IFUs and stack by group_id ────────────────────────────────────────
#
# OB folder names use hyphens for the time component
# (KMOS.2026-01-13T03-42-28.829) while dp_ids use colons
# (KMOS.2026-01-13T03:42:28.829). ob_to_dpid() converts between them.
#
# Strategy:
#   For each (OB, band) on disk → convert OB name to dp_id →
#   look up (dp_id, ifu) in catalog to get group_id →
#   collect all individual exposure spectra under that group_id.
#   Final stacked spectrum = nanmedian over all collected spectra.

print("Scanning exposures and grouping by group_id …")

# group_data[(group_id, band)] collects everything for one galaxy+band stack
group_data = defaultdict(lambda: dict(
    spectra=[], obs=set(), wave=None,
    z=None, z_type='', z_quality=np.nan,
    cube=None, ha_image=None,
    ctr_ha=None, ctr_geom=None, fallback=False,
    _best_ha_snr=-np.inf,
))

exposures = find_exposures()
n_matched = 0
n_skipped = 0

for (ob, band), paths in sorted(exposures.items()):
    dpid = ob_to_dpid(ob)

    # Find all IFUs active in any file of this OB
    ifu_set = set()
    for p in paths:
        with fits.open(p) as h:
            ifu_set.update(active_ifus(h))

    for ifu in sorted(ifu_set):
        group_id = dpid_ifu_to_group.get((dpid, ifu))
        if group_id is None:
            n_skipped += 1
            continue
        n_matched += 1
        key = (group_id, band)
        g   = group_data[key]

        spectra_this_ob = []
        wave_this_ob    = None
        cube_data       = None
        ctr_geom        = None
        ctr_ha          = None
        ha_image        = None
        fallback        = False

        for p in paths:
            with fits.open(p) as h:
                if ifu not in active_ifus(h):
                    continue
                cube = np.array(h[ifu].data)

                if wave_this_ob is None:
                    wave_this_ob = wavelength_axis(h[ifu].header)
                    cube_data    = cube
                    ctr_geom     = ifu_center(cube)

                    # Fetch redshift from first encounter
                    if g['z'] is None:
                        z, z_type, z_quality = get_z_meta(h, ifu)
                        if z is not None:
                            g.update(z=z, z_type=z_type, z_quality=z_quality)

                    z = g['z']
                    if z is not None:
                        ctr_raw, ha_image = ha_peak_center(cube, wave_this_ob, z)
                        offset = np.hypot(ctr_raw[0] - ctr_geom[0],
                                          ctr_raw[1] - ctr_geom[1])
                        if offset > MAX_CENTER_OFFSET:
                            ctr_ha   = ctr_geom
                            fallback = True
                        else:
                            ctr_ha = ctr_raw
                    else:
                        ctr_ha = ctr_geom

                ctr = ctr_ha if ctr_ha is not None else ifu_center(cube)
                _, s = aperture_spectrum(h, ifu, ctr)
                s_ref = np.nanmedian(s)
                if np.isfinite(s_ref) and s_ref != 0:
                    s = s / s_ref
                spectra_this_ob.append(s)

        if not spectra_this_ob or wave_this_ob is None:
            continue

        # Keep wave from first OB (grids are identical within a band+config)
        if g['wave'] is None:
            g['wave'] = wave_this_ob

        # Use spatial info from the OB with the best Hα SNR
        z = g['z']
        if z is not None and cube_data is not None:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                med_ob = np.nanmedian(np.vstack(spectra_this_ob), axis=0)
            snr_ob = ha_snr(g['wave'], med_ob, z)
            if np.isfinite(snr_ob) and snr_ob > g['_best_ha_snr']:
                g.update(
                    _best_ha_snr=snr_ob,
                    cube=cube_data, ha_image=ha_image,
                    ctr_ha=ctr_ha, ctr_geom=ctr_geom,
                    fallback=fallback,
                )

        g['spectra'].extend(spectra_this_ob)
        g['obs'].add(ob)

print(f"  IFU×OB matched to catalog:  {n_matched}")
print(f"  IFU×OB not in catalog:      {n_skipped}")
print(f"  Unique (group_id, band) stacks: {len(group_data)}")

Scanning exposures and grouping by group_id …


C:\Users\kreuter\AppData\Local\Temp\ipykernel_17640\1699198028.py:86: RuntimeWarning: All-NaN slice encountered
  s_ref = np.nanmedian(s)


  IFU×OB matched to catalog:  317
  IFU×OB not in catalog:      0
  Unique (group_id, band) stacks: 37


In [ ]:
# ── Build final stacked entries and compute SNRs ────────────────────────────────
LINE_DEFS = {
    'ha':   (HA_REST,  SNR_WIN_HALF, SNR_CONT_IN, SNR_CONT_OUT),
    'hb':   (0.48627,  0.0020, 0.005, 0.018),
    'oiii': (0.50082,  0.0020, 0.005, 0.018),
    'sii':  (0.67163,  0.0020, 0.005, 0.018),
}

all_stacked = []
_skipped_no_z = []

for (group_id, band), g in group_data.items():
    if not g['spectra'] or g['wave'] is None:
        continue
    if g.get('z') is None:
        _skipped_no_z.append((group_id, band, len(g['obs']), len(g['spectra'])))
        continue
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        med = np.nanmedian(np.vstack(g['spectra']), axis=0)

    snrs = {}
    for lkey, (lrest, wh, ci, co) in LINE_DEFS.items():
        snrs[lkey] = line_snr(g['wave'], med, g['z'], lrest,
                              win_half=wh, cont_in=ci, cont_out=co)

    all_stacked.append(dict(
        group_id  = group_id,
        band      = band,
        z         = g['z'],
        z_type    = g['z_type'],
        z_quality = g['z_quality'],
        wave      = g['wave'],
        spectra   = g['spectra'],
        med       = med,
        n_obs     = len(g['obs']),
        n_exp     = len(g['spectra']),
        cube      = g.get('cube'),
        ha_image  = g.get('ha_image'),
        ctr_ha    = g.get('ctr_ha'),
        ctr_geom  = g.get('ctr_geom'),
        fallback  = g.get('fallback', False),
        **{f'snr_{k}': v for k, v in snrs.items()},
    ))

# Sort by Hα SNR (NaN last)
all_stacked.sort(
    key=lambda x: x['snr_ha'] if np.isfinite(x.get('snr_ha', np.nan)) else -np.inf,
    reverse=True
)

n_ha  = sum(1 for e in all_stacked if np.isfinite(e.get('snr_ha', np.nan))  and e['snr_ha']  >= SNR_MIN)
n_hb  = sum(1 for e in all_stacked if np.isfinite(e.get('snr_hb', np.nan))  and e['snr_hb']  >= SNR_MIN)
n_oi  = sum(1 for e in all_stacked if np.isfinite(e.get('snr_oiii', np.nan)) and e['snr_oiii'] >= SNR_MIN)
n_si  = sum(1 for e in all_stacked if np.isfinite(e.get('snr_sii', np.nan))  and e['snr_sii']  >= SNR_MIN)
print(f"Stacked galaxies: {len(all_stacked)}")
print(f"  Hα   SNR ≥ {SNR_MIN}: {n_ha}")
print(f"  Hβ   SNR ≥ {SNR_MIN}: {n_hb}")
print(f"  [OIII] SNR ≥ {SNR_MIN}: {n_oi}")
print(f"  [SII]  SNR ≥ {SNR_MIN}: {n_si}")

# ── DIAGNOSTIC: full listing by n_obs ─────────────────────────────────────────
print(f"\nAll {len(all_stacked)} stacked entries (sorted by n_obs):")
print(f"{'gid':>6}  {'band':5}  {'z':>7}  {'z_type':>7}  {'n_obs':>5}  "
      f"{'Hα':>6}  {'Hβ':>6}  {'[OIII]':>7}  {'[SII]':>6}  {'fb':>4}")
print("-" * 72)
def _f(v): return f"{v:.1f}" if np.isfinite(v) else "NaN"
for e in sorted(all_stacked, key=lambda x: x['n_obs'], reverse=True):
    print(f"{e['group_id']:>6}  {e['band']:5}  {e['z']:>7.4f}  {e.get('z_type','?'):>7}  "
          f"{e['n_obs']:>5}  {_f(e['snr_ha']):>6}  {_f(e['snr_hb']):>6}  "
          f"{_f(e['snr_oiii']):>7}  {_f(e['snr_sii']):>6}  "
          f"{'YES' if e.get('fallback') else '':>4}")

if _skipped_no_z:
    print(f"\n⚠ Skipped — no redshift in CSV ({len(_skipped_no_z)} entries):")
    print(f"{'gid':>6}  {'band':5}  {'n_obs':>5}  {'n_exp':>5}")
    for gid, band, nobs, nexp in sorted(_skipped_no_z, key=lambda x: x[2], reverse=True):
        print(f"{gid:>6}  {band:5}  {nobs:>5}  {nexp:>5}")

In [ ]:
# ── Ranking table ─────────────────────────────────────────────────────────────
def _fmt(v):
    return f"{v:.1f}" if np.isfinite(v) else "—"

print(f"{'#':>3}  {'gid':>6}  {'band':5}  {'z':>7}  {'type':>7}  {'q':>3}  "
      f"{'N_obs':>5}  {'N_exp':>5}  {'Hα':>6}  {'Hβ':>6}  {'[OIII]':>7}  {'[SII]':>6}")
print("-" * 88)
_spec_stacked = [e for e in all_stacked if e.get("z_type") in ("spec", "cluster")]
for i, e in enumerate(_spec_stacked):
    q_s = f"{e['z_quality']:.0f}" if np.isfinite(e.get('z_quality', np.nan)) else "—"
    print(f"{i+1:>3}  {e['group_id']:>6}  {e['band']:5}  {e['z']:>7.4f}  "
          f"{e['z_type']:>7}  {q_s:>3}  "
          f"{e['n_obs']:>5}  {e['n_exp']:>5}  "
          f"{_fmt(e['snr_ha']):>6}  {_fmt(e['snr_hb']):>6}  "
          f"{_fmt(e['snr_oiii']):>7}  {_fmt(e['snr_sii']):>6}")

## Line-window gallery

Each row shows one stacked galaxy with **four** emission-line zoom panels:
**Hα** · **Hβ** · **[OIII]λ5007** · **[SII]λ6716**.

Individual exposures are plotted in light grey; the stacked median is the
coloured curve.  Panels whose line falls outside the detector wavelength range
are labelled *out of range*.  Galaxies are ordered by Hα SNR.

In [ ]:
# Line zoom definitions: (key, rest_µm, label, line_color, window_half_µm, companions)
# companions: list of (label, rest_µm, color, lw, ls)
ZOOM_PANELS = [
    (
        'ha', HA_REST, r'H$\alpha$', '#d62728', ZOOM_HALF,
        [('[NII]6548', 0.65480, '#ff7f0e', 0.8, ':'),
         ('[NII]6583', 0.65835, '#ff7f0e', 1.1, '--')],
    ),
    (
        'hb', 0.48627, r'H$\beta$', '#1f77b4', HB_ZOOM_HALF,
        [('[OIII]4959', 0.49594, '#7fdbff', 0.8, ':')],
    ),
    (
        'oiii', 0.50082, r'[OIII]$\lambda$5007', '#17becf', ZOOM_HALF,
        [('[OIII]4959', 0.49594, '#7fdbff', 0.8, ':'),
         (r'H$\beta$', 0.48627, '#1f77b4', 0.8, '--')],
    ),
    (
        'sii', 0.67163, r'[SII]$\lambda$6716', '#9467bd', ZOOM_HALF,
        [(r'H$\alpha$', HA_REST, '#d62728', 1.2, '--'),
         ('[NII]6583', 0.65835, '#ff7f0e', 0.9, ':'),
         ('[SII]6731', 0.67308, '#c5b0d8', 0.9, ':')],
    ),
]

detected = [
    e for e in all_stacked
    if np.isfinite(e.get('snr_ha', np.nan)) and e['snr_ha'] >= SNR_MIN
    and e.get('z_type') in ('spec', 'cluster')
]

if not detected:
    print(f"No galaxies with Hα SNR ≥ {SNR_MIN}.")
else:
    ncols = 4  # one column per emission-line window
    nrows = len(detected)
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(ncols * 4.8, nrows * 2.8),
        constrained_layout=True,
    )
    if nrows == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(
        f"KMOS Stacked Galaxies — Line Windows  "
        f"({len(detected)} galaxies with Hα SNR ≥ {SNR_MIN}, spec/cluster z)",
        fontsize=12, fontweight='bold', y=1.002,
    )

    # Column headers
    for ci, (_, _, label, lc, _, _) in enumerate(ZOOM_PANELS):
        axes[0, ci].set_title(label, fontsize=10, fontweight='bold', color=lc, pad=8)

    for ri, (e, color) in enumerate(zip(detected, PALETTE)):
        wave, z = e['wave'], e['z']

        for ci, (lkey, lrest, label, lc, zhalf, companions) in enumerate(ZOOM_PANELS):
            ax   = axes[ri, ci]
            lam  = lrest * (1.0 + z)
            wlo  = lam - zhalf
            whi  = lam + zhalf
            mask = (wave >= wlo) & (wave <= whi)

            if not mask.any() or not (wave.min() <= lam <= wave.max()):
                ax.text(0.5, 0.5, 'out of range',
                        ha='center', va='center', transform=ax.transAxes,
                        fontsize=8, color='#aaa', style='italic')
                ax.set_xlim(wlo * 1e4, whi * 1e4)
                ax.tick_params(labelsize=6)
                if ri < nrows - 1:
                    ax.set_xticklabels([])
                continue

            # Individual exposures
            for s in e['spectra']:
                ax.plot(wave[mask] * 1e4, s[mask],
                        color='#d0d0d0', lw=0.4, alpha=0.6, zorder=1)
            # Stacked median
            ax.plot(wave[mask] * 1e4, e['med'][mask],
                    color=color, lw=1.8, zorder=3)

            # Primary line marker
            ax.axvline(lam * 1e4, color=lc, lw=1.3, ls='--', alpha=0.85, zorder=2)
            ax.text(lam * 1e4, 0.97, label,
                    transform=ax.get_xaxis_transform(),
                    rotation=90, ha='right', va='top', fontsize=6, color=lc,
                    bbox=dict(fc='white', ec='none', alpha=0.6, pad=1))

            # Companion lines
            for cname, crest, clc, clw, cls in companions:
                co_lam = crest * (1.0 + z)
                if wlo <= co_lam <= whi:
                    ax.axvline(co_lam * 1e4, color=clc, lw=clw, ls=cls,
                               alpha=0.75, zorder=2)
                    ax.text(co_lam * 1e4, 0.97, cname,
                            transform=ax.get_xaxis_transform(),
                            rotation=90, ha='right', va='top',
                            fontsize=5.5, color=clc,
                            bbox=dict(fc='white', ec='none', alpha=0.6, pad=1))

            # Y limits
            vis = e['med'][mask]
            fv  = vis[np.isfinite(vis)]
            if fv.size:
                ylo = np.nanpercentile(fv, 2)
                yhi = np.nanmax(fv)
                pad = 0.18 * max(yhi - ylo, 0)
                ax.set_ylim(ylo - pad, yhi + pad)

            ax.set_xlim(wlo * 1e4, whi * 1e4)
            ax.tick_params(labelsize=6)
            if ri < nrows - 1:
                ax.set_xticklabels([])
            if ci == 0:
                fb = "  [fb]" if e.get('fallback') else ""
                ax.set_ylabel(
                    f"gid {e['group_id']}  z={z:.4f} [{e['z_type']}]{fb}\n"
                    f"{e['n_obs']} OBs · {e['n_exp']} exp",
                    fontsize=7, labelpad=4
                )
            # SNR annotation
            snr_v = e.get(f'snr_{lkey}', np.nan)
            if np.isfinite(snr_v):
                ax.text(0.98, 0.96, f"SNR={snr_v:.1f}",
                        transform=ax.transAxes, ha='right', va='top',
                        fontsize=6.5, color='#333',
                        bbox=dict(fc='white', ec='none', alpha=0.7, pad=1))

        # X label on bottom row only
        if ri == nrows - 1:
            for ci in range(ncols):
                axes[ri, ci].set_xlabel(r'$\lambda_{\rm obs}$  (Å)', fontsize=7)

    plt.savefig('kmos_stacked_gallery.pdf', bbox_inches='tight')
    plt.show()
    print('Saved → kmos_stacked_gallery.pdf')

## Detail view — full spectrum + spatial maps

For the best stacked galaxies: the top panel shows the **full spectrum** with
every emission line in band annotated; the middle row shows four
**zoom panels** (Hα · Hβ · [OIII]5007 · [SII]6716); the bottom row shows
the **white-light** and narrow-band spatial maps.

In [ ]:
from matplotlib.patches import Rectangle

def _apt_rect(cy, cx, box=APERTURE, **kw):
    h2 = box // 2
    return Rectangle((cx - h2 - 0.5, cy - h2 - 0.5), box, box,
                      linewidth=1.4, fill=False, **kw)

top = [
    e for e in all_stacked
    if np.isfinite(e.get('snr_ha', np.nan)) and e['snr_ha'] >= SNR_MIN
    and e.get('z_type') in ('spec', 'cluster')
][:TOP_DETAIL]

if not top:
    print('No targets to display.')
else:
    for e, color in zip(top, PALETTE):
        wave, z = e['wave'], e['z']
        ha_obs  = HA_REST * (1.0 + z)
        fb = "  [fallback ctr]" if e.get('fallback') else ""

        fig = plt.figure(figsize=(16, 9), constrained_layout=True)
        fig.suptitle(
            f"group_id {e['group_id']}  ·  {e['band']} band  ·  "
            f"z = {z:.4f} [{e['z_type']}]{fb}  ·  "
            f"{e['n_obs']} OBs · {e['n_exp']} exposures stacked  ·  "
            f"Hα SNR = {e['snr_ha']:.1f}",
            fontsize=10, fontweight='bold',
        )

        outer = gridspec.GridSpec(3, 1, figure=fig,
                                  height_ratios=[2.0, 1.5, 1.5], hspace=0.42)

        # ── Row 0: full-spectrum overview ──────────────────────────────────────
        ax_full = fig.add_subplot(outer[0])

        overview_values = []

        for spectrum in e['spectra']:
            valid = np.isfinite(wave) & np.isfinite(spectrum)
            if valid.any():
                ax_full.plot(
                    wave[valid] * 1e4,
                    spectrum[valid],
                    color='#d0d0d0',
                    lw=0.35,
                    alpha=0.6,
                    zorder=1,
                )
                overview_values.append(spectrum[valid])

        valid_med = np.isfinite(wave) & np.isfinite(e['med'])
        if valid_med.any():
            ax_full.plot(
                wave[valid_med] * 1e4,
                e['med'][valid_med],
                color=color,
                lw=1.6,
                zorder=3,
                label='stacked median',
            )
            overview_values.append(e['med'][valid_med])

        if overview_values:
            med_values = e['med'][np.isfinite(e['med'])]
            if med_values.size:
                ymin = np.nanpercentile(med_values, 2)
                ymax = np.nanmax(med_values)
                padding = 0.08 * max(ymax - ymin, 1e-12)
                ax_full.set_ylim(ymin - padding, ymax + padding)

        for lname, (lrest, lc) in EMISSION_LINES.items():
            line_wave = lrest * (1 + z)
            if wave.min() <= line_wave <= wave.max():
                xline = line_wave * 1e4
                ax_full.axvline(xline, color=lc, lw=1.0, ls='--', alpha=0.7)
                ax_full.text(
                    xline, 0.97, lname,
                    transform=ax_full.get_xaxis_transform(),
                    rotation=90, ha='right', va='top',
                    fontsize=7, color=lc,
                    bbox=dict(fc='white', ec='none', alpha=0.6, pad=1),
                )

        ax_full.set_xlim(wave.min() * 1e4, wave.max() * 1e4)
        ax_full.set_xlabel(r'$\lambda_{\rm obs}$  (Å)', fontsize=8)
        ax_full.set_ylabel('flux', fontsize=8)
        ax_full.set_title('Full-spectrum overview', fontsize=8, pad=3)
        ax_full.tick_params(labelsize=7)
        ax_full.legend(fontsize=7, loc='upper left', frameon=False)

        # ── Row 1: four line zoom panels ───────────────────────────────────────
        inner_zoom = gridspec.GridSpecFromSubplotSpec(
            1, 4, subplot_spec=outer[1], wspace=0.35)

        for ci, (lkey, lrest, label, lc, zhalf, companions) in enumerate(ZOOM_PANELS):
            ax = fig.add_subplot(inner_zoom[ci])
            lam = lrest * (1.0 + z)
            wlo = lam - zhalf
            whi = lam + zhalf
            mask = (wave >= wlo) & (wave <= whi)

            if not mask.any() or not (wave.min() <= lam <= wave.max()):
                ax.text(0.5, 0.5, 'out of range',
                        ha='center', va='center', transform=ax.transAxes,
                        fontsize=8, color='#aaa', style='italic')
                ax.set_title(label, fontsize=8, color=lc, pad=3)
                continue

            for s in e['spectra']:
                ax.plot(wave[mask] * 1e4, s[mask],
                        color='#d0d0d0', lw=0.4, alpha=0.6, zorder=1)
            ax.plot(wave[mask] * 1e4, e['med'][mask],
                    color=color, lw=2.0, zorder=3)

            ax.axvline(lam * 1e4, color=lc, lw=1.5, ls='--', alpha=0.85, zorder=2)
            ax.text(lam * 1e4, 0.97, label,
                    transform=ax.get_xaxis_transform(),
                    rotation=90, ha='right', va='top', fontsize=7, color=lc,
                    bbox=dict(fc='white', ec='none', alpha=0.6, pad=1))

            for cname, crest, clc, clw, cls in companions:
                co_lam = crest * (1.0 + z)
                if wlo <= co_lam <= whi:
                    ax.axvline(co_lam * 1e4, color=clc, lw=clw, ls=cls,
                               alpha=0.75, zorder=2)
                    ax.text(co_lam * 1e4, 0.97, cname,
                            transform=ax.get_xaxis_transform(),
                            rotation=90, ha='right', va='top',
                            fontsize=6, color=clc,
                            bbox=dict(fc='white', ec='none', alpha=0.6, pad=1))

            vis = e['med'][mask]
            fv  = vis[np.isfinite(vis)]
            if fv.size:
                ylo = np.nanpercentile(fv, 2)
                yhi = np.nanmax(fv)
                pad = 0.18 * max(yhi - ylo, 0)
                ax.set_ylim(ylo - pad, yhi + pad)

            ax.set_xlim(wlo * 1e4, whi * 1e4)
            ax.tick_params(labelsize=7)
            ax.set_xlabel(r'$\lambda_{\rm obs}$  (Å)', fontsize=7)
            snr_v = e.get(f'snr_{lkey}', np.nan)
            snr_s = f'SNR={snr_v:.1f}' if np.isfinite(snr_v) else ''
            ax.set_title(f'{label}   {snr_s}', fontsize=8, color=lc, pad=3)

        # ── Row 2: spatial maps ────────────────────────────────────────────────
        inner_maps = gridspec.GridSpecFromSubplotSpec(
            1, 5, subplot_spec=outer[2], wspace=0.35)

        map_lines = [
            ('white-light', None,    None,   'viridis'),
            (r'H$\alpha$',  HA_REST, '#d62728', 'inferno'),
            (r'H$\beta$',   0.48627, '#1f77b4',  'plasma'),
            ('[OIII]5007',  0.50082, '#17becf',  'cividis'),
            ('[SII]6716',   0.67163, '#9467bd',  'hot'),
        ]

        cube = e.get('cube')
        if cube is not None:
            valid_cube = _erode_valid(np.isfinite(cube), EDGE_MARGIN)
            wl_im = np.nansum(np.where(valid_cube, cube, np.nan), axis=0)
            cy_ha, cx_ha   = (e['ctr_ha']   or ifu_center(cube))
            cy_geo, cx_geo = (e['ctr_geom'] or ifu_center(cube))
            moved = (cy_ha, cx_ha) != (cy_geo, cx_geo)

            for mi, (map_label, map_rest, marker_c, cmap) in enumerate(map_lines):
                ax_m = fig.add_subplot(inner_maps[mi])

                if map_rest is None:
                    im = wl_im
                else:
                    im = narrowband_image(cube, wave, z, map_rest)

                if im is None or not np.isfinite(im).any():
                    ax_m.text(0.5, 0.5, 'out of range',
                              ha='center', va='center',
                              transform=ax_m.transAxes,
                              fontsize=7, color='#aaa', style='italic')
                    ax_m.set_title(map_label, fontsize=7, pad=3)
                    ax_m.tick_params(labelsize=5)
                    continue

                fin = im[np.isfinite(im)]
                ax_m.imshow(im, origin='lower', aspect='equal', cmap=cmap,
                            vmin=np.nanpercentile(fin, 5),
                            vmax=np.nanpercentile(fin, 97))

                if moved:
                    ax_m.plot(cx_geo, cy_geo, 'x', ms=5, mew=1.2,
                              color='#ffff80', alpha=0.6, zorder=4)
                ec = marker_c or 'red'
                ax_m.plot(cx_ha, cy_ha, '+', ms=6, mew=1.5, color=ec, zorder=5)
                ax_m.add_patch(_apt_rect(cy_ha, cx_ha, edgecolor=ec, zorder=5))
                ax_m.set_title(map_label, fontsize=7, pad=3)
                ax_m.tick_params(labelsize=5)
        else:
            for mi in range(5):
                ax_m = fig.add_subplot(inner_maps[mi])
                ax_m.set_visible(False)

        fname = f"kmos_stacked_detail_gid{e['group_id']}_{e['band']}.pdf"
        plt.savefig(fname, bbox_inches='tight')
        plt.show()
        print(f"Saved → {fname}")

## Full-band spectra — all stacked galaxies

Full wavelength range, ordered by Hα SNR.
Individual exposures in grey; stacked nanmedian coloured.
Dashed lines = emission lines at catalogue redshift.
Grey shading = atmospheric absorption bands.

In [ ]:
# ── Telluric correction quality ────────────────────────────────────────────────
# For OBs where both _rv.fits and _rv_tell.fits are present:
# compare aperture spectra and Hα SNR before/after correction.

TELL_SUFFIX   = "_s_i_bkg2d_rv_tell.fits"
NOTELL_SUFFIX = "_s_i_bkg2d_rv.fits"
_ATMOS = [
    (0.685, 0.695), (0.715, 0.735), (0.758, 0.772), (0.805, 0.840),
    (0.880, 0.990), (1.060, 1.110), (1.130, 1.160), (1.270, 1.320),
]
def _shade(ax, wlo, whi):
    for lo, hi in _ATMOS:
        if hi < wlo or lo > whi: continue
        ax.axvspan(max(lo, wlo) * 1e4, min(hi, whi) * 1e4, alpha=0.10, color='#888', lw=0)

# ── Find paired files ──────────────────────────────────────────────────────────
pairs = []   # (ob, tell_path, notell_path)
for ob_dir in sorted(glob.glob(os.path.join(BASE, 'KMOS.*'))):
    ob      = os.path.basename(ob_dir)
    sci_dir = os.path.join(ob_dir, 'science')
    tells   = {os.path.basename(p).replace(TELL_SUFFIX,   ''): p
               for p in glob.glob(os.path.join(sci_dir, f'*{TELL_SUFFIX}'))}
    notells = {os.path.basename(p).replace(NOTELL_SUFFIX, ''): p
               for p in glob.glob(os.path.join(sci_dir, f'*{NOTELL_SUFFIX}'))}
    for stem, tp in tells.items():
        if stem in notells:
            pairs.append((ob, tp, notells[stem]))

n_obs_with_pairs = len(set(ob for ob, _, _ in pairs))
print(f"OBs with both files: {n_obs_with_pairs}  |  file pairs: {len(pairs)}")

if not pairs:
    print("No paired files found — nothing to compare.")
else:
    # ── Extract spectra for every IFU with a known spectroscopic/cluster redshift ──
    rows = []
    for ob, tell_path, notell_path in pairs:
        dpid = ob_to_dpid(ob)
        with fits.open(tell_path) as h_t, fits.open(notell_path) as h_n:
            common = set(active_ifus(h_t)) & set(active_ifus(h_n))
            for ifu in sorted(common):
                hdr0  = h_t[0].header
                field = (hdr0.get("ESO OBS TARG NAME") or "").strip()
                tid   = str(hdr0.get(f"ESO OCS ARM{ifu} NAME") or "").strip()
                z     = _Z_TABLE.get((field, tid)) if field and tid else None
                if z is None:
                    continue
                if _Z_TYPE.get((field, tid), '') not in ('spec', 'cluster'):
                    continue

                wave = wavelength_axis(h_t[ifu].header)
                cube_t = np.array(h_t[ifu].data)

                # centroid on Hα in the tell-corrected cube (better S/N there)
                ctr, _ = ha_peak_center(cube_t, wave, z)
                offset = np.hypot(ctr[0] - ifu_center(cube_t)[0],
                                  ctr[1] - ifu_center(cube_t)[1])
                if offset > MAX_CENTER_OFFSET:
                    ctr = ifu_center(cube_t)

                _, s_tell   = aperture_spectrum(h_t, ifu, ctr)
                _, s_notell = aperture_spectrum(h_n, ifu, ctr)

                # normalise to the no-tell median so scales are comparable
                ref = np.nanmedian(s_notell)
                if not np.isfinite(ref) or ref == 0:
                    continue
                s_tell   = s_tell   / ref
                s_notell = s_notell / ref

                snr_t = ha_snr(wave, s_tell,   z)
                snr_n = ha_snr(wave, s_notell, z)
                gid   = dpid_ifu_to_group.get((dpid, ifu))
                rows.append(dict(ob=ob, ifu=ifu, z=z, gid=gid,
                                 wave=wave, s_tell=s_tell, s_notell=s_notell,
                                 snr_tell=snr_t, snr_notell=snr_n))

    if not rows:
        print("No paired IFUs with known redshift found.")
    else:
        # ── Summary table ──────────────────────────────────────────────────────
        has_both = [r for r in rows
                    if np.isfinite(r['snr_tell']) and np.isfinite(r['snr_notell'])]
        improved = sum(1 for r in has_both if r['snr_tell'] > r['snr_notell'])
        gains    = [r['snr_tell'] - r['snr_notell'] for r in has_both]
        med_gain = np.nanmedian(gains) if gains else np.nan

        rows_sorted = sorted(
            rows, key=lambda r: r['snr_tell'] if np.isfinite(r.get('snr_tell', np.nan)) else -np.inf,
            reverse=True)

        print(f"\n{'OB':35s}  {'IFU':>4}  {'z':>7}  {'gid':>6}  "
              f"{'SNR (no-tell)':>13}  {'SNR (tell)':>10}  {'gain':>6}")
        print("-" * 90)
        for r in rows_sorted:
            sn = _fmt(r['snr_notell']); st = _fmt(r['snr_tell'])
            g  = (f"{r['snr_tell']-r['snr_notell']:+.1f}"
                  if np.isfinite(r['snr_tell']) and np.isfinite(r['snr_notell']) else "—")
            print(f"{r['ob'][:35]:35s}  {r['ifu']:>4}  {r['z']:>7.4f}  "
                  f"{str(r['gid'] or '—'):>6}  {sn:>13}  {st:>10}  {g:>6}")

        print(f"\nResult: {improved}/{len(has_both)} IFUs improved  |  "
              f"median Hα SNR gain = {med_gain:+.2f}"
              if has_both else "\n(No IFUs with finite SNR in both versions)")

        # ── Comparison plots ───────────────────────────────────────────────────
        plot_rows = rows_sorted[:8]   # top 8 by tell-corrected SNR

        fig, axes = plt.subplots(len(plot_rows), 3,
                                  figsize=(18, len(plot_rows) * 2.8),
                                  constrained_layout=True)
        if len(plot_rows) == 1:
            axes = axes[np.newaxis, :]
        fig.suptitle(
            f"Telluric correction comparison  "
            f"({improved}/{len(has_both)} IFUs improved, median Hα gain {med_gain:+.2f})",
            fontsize=11, fontweight='bold', y=1.002)

        for ri, r in enumerate(plot_rows):
            ax_ov   = axes[ri, 0]
            ax_rat  = axes[ri, 1]
            ax_zoom = axes[ri, 2]
            wave = r['wave']
            wlo_um = float(wave.min()); whi_um = float(wave.max())
            color  = PALETTE[ri % len(PALETTE)]
            z      = r['z']

            # ── left: overlay ─────────────────────────────────────────────────
            ax_ov.plot(wave * 1e4, r['s_notell'],
                       color='#aaaaaa', lw=0.9, alpha=0.9, zorder=2,
                       label='no telluric corr.')
            ax_ov.plot(wave * 1e4, r['s_tell'],
                       color=color, lw=1.5, alpha=0.95, zorder=3,
                       label='telluric corr.')
            ax_ov.axhline(0, color='#ccc', lw=0.4, zorder=0)
            _shade(ax_ov, wlo_um, whi_um)
            for lname, (lrest, lc) in EMISSION_LINES.items():
                lobs = lrest * (1 + z)
                if wlo_um <= lobs <= whi_um:
                    ax_ov.axvline(lobs * 1e4, color=lc, lw=0.7, ls='--', alpha=0.55, zorder=2)

            vis = np.concatenate([r['s_tell'][np.isfinite(r['s_tell'])],
                                   r['s_notell'][np.isfinite(r['s_notell'])]])
            if vis.size:
                ylo = np.nanpercentile(vis, 2); yhi = np.nanmax(vis)
                pad = 0.10 * max(yhi - ylo, 0)
                ax_ov.set_ylim(ylo - pad, yhi + pad)
            ax_ov.set_xlim(wlo_um * 1e4, whi_um * 1e4)
            ax_ov.legend(fontsize=7, loc='upper right', frameon=False)
            ax_ov.set_ylabel('flux (norm.)', fontsize=8)
            ax_ov.set_xlabel(r'$\lambda_{\rm obs}$ (Å)', fontsize=8)
            ax_ov.tick_params(labelsize=7)
            sn = _fmt(r['snr_notell']); st = _fmt(r['snr_tell'])
            g  = (f"  [+{r['snr_tell']-r['snr_notell']:.1f}]"
                  if np.isfinite(r['snr_tell']) and np.isfinite(r['snr_notell']) else "")
            ax_ov.set_title(
                f"IFU {r['ifu']}  ·  {r['ob'][5:21]}  ·  z={z:.4f}  ·  "
                f"Hα SNR: {sn} → {st}{g}",
                fontsize=8, pad=4)

            # ── right: ratio ──────────────────────────────────────────────────
            with np.errstate(divide='ignore', invalid='ignore'):
                ratio = np.where(np.abs(r['s_notell']) > 0.05,
                                 r['s_tell'] / r['s_notell'], np.nan)
            ax_rat.plot(wave * 1e4, ratio, color='#444', lw=0.8, zorder=2)
            ax_rat.axhline(1.0, color='#d62728', lw=1.0, ls='--', alpha=0.7, zorder=3,
                           label='ratio = 1 (no change)')
            _shade(ax_rat, wlo_um, whi_um)
            ratio_ok = ratio[np.isfinite(ratio)]
            if ratio_ok.size:
                rlo = np.nanpercentile(ratio_ok, 2)
                rhi = np.nanpercentile(ratio_ok, 98)
                pad = 0.15 * max(rhi - rlo, 0)
                ax_rat.set_ylim(max(rlo - pad, -0.5), rhi + pad)
            ax_rat.set_xlim(wlo_um * 1e4, whi_um * 1e4)
            ax_rat.set_ylabel('tell / no-tell', fontsize=8)
            ax_rat.set_xlabel(r'$\lambda_{\rm obs}$ (Å)', fontsize=8)
            ax_rat.tick_params(labelsize=7)
            ax_rat.set_title('Correction ratio  (grey = atmos. bands)', fontsize=8, pad=4)
            ax_rat.legend(fontsize=7, frameon=False)

            # ── zoom: Hα + [NII] ─────────────────────────────────────────────
            _ha_obs  = HA_REST * (1 + z)
            _lo_um   = _ha_obs - 0.022
            _hi_um   = _ha_obs + 0.018
            _wmask   = (wave >= _lo_um) & (wave <= _hi_um)
            ax_zoom.plot(wave[_wmask] * 1e4, r['s_notell'][_wmask],
                         color='#aaaaaa', lw=1.2, alpha=0.9, label='no telluric corr.')
            ax_zoom.plot(wave[_wmask] * 1e4, r['s_tell'][_wmask],
                         color=color, lw=1.5, alpha=0.95, label='telluric corr.')
            ax_zoom.axhline(0, color='#ccc', lw=0.4, zorder=0)
            for lname, (lrest, lc) in EMISSION_LINES.items():
                lobs_um = lrest * (1 + z)
                if _lo_um <= lobs_um <= _hi_um:
                    ax_zoom.axvline(lobs_um * 1e4, color=lc, lw=0.8, ls='--', alpha=0.6, zorder=2)
                    ax_zoom.text(lobs_um * 1e4, 0.97, lname,
                                 transform=ax_zoom.get_xaxis_transform(),
                                 rotation=90, ha='right', va='top',
                                 fontsize=6, color=lc, clip_on=True)
            _vis_z = np.concatenate([r['s_tell'][_wmask & np.isfinite(r['s_tell'])],
                                     r['s_notell'][_wmask & np.isfinite(r['s_notell'])]])
            if _vis_z.size:
                _ylo_z = np.nanpercentile(_vis_z, 1)
                _yhi_z = np.nanmax(_vis_z)
                _pad_z = 0.12 * max(_yhi_z - _ylo_z, 1e-30)
                ax_zoom.set_ylim(_ylo_z - _pad_z, _yhi_z + _pad_z)
            ax_zoom.set_xlim(_lo_um * 1e4, _hi_um * 1e4)
            ax_zoom.set_xlabel(r'$\lambda_{\rm obs}$ (Å)', fontsize=8)
            ax_zoom.set_ylabel('flux (norm.)', fontsize=8)
            ax_zoom.tick_params(labelsize=7)
            ax_zoom.set_title('Hα + [NII] zoom', fontsize=8, pad=4)
            ax_zoom.legend(fontsize=6, loc='upper right', frameon=False)
        plt.savefig('kmos_tell_quality.pdf', bbox_inches='tight')
        plt.show()
        print('Saved → kmos_tell_quality.pdf')